# 07 — Lightning Flash (Tabular Classification)

This notebook is self-contained for a fresh Google Colab runtime. Run the cells from top to bottom. It clones the team repository, creates the processed dataset, builds a separate Python 3.10 environment for Lightning Flash 0.8.2, and runs the Flash TabNet classifier.

**Important:** Flash 0.8.2 is an older release. This notebook records the actual outcome; it does not invent metrics. If training fails, retain the complete traceback and report the framework attempt as unsuccessful rather than substituting another framework.

## 1. Clone the project repository

In [1]:
import os
from pathlib import Path

REPO = Path("/content/loan-default-mlbd")

if not REPO.exists():
    !git clone https://github.com/kratipatel/loan-default-mlbd.git {REPO}

%cd /content/loan-default-mlbd

print("Repository:", os.getcwd())
print("src exists:", Path("src").is_dir())
print("prepare_data exists:", Path("src/prepare_data.py").is_file())

Cloning into '/content/loan-default-mlbd'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 107 (delta 38), reused 66 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 1.34 MiB | 16.53 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/loan-default-mlbd
Repository: /content/loan-default-mlbd
src exists: True
prepare_data exists: True


## 2. Install the dependencies for dataset preparation
This uses the normal Colab Python only for the team's preprocessing script. Flash itself will run in the separate Python 3.10 environment.

In [2]:
%pip install -q pandas numpy pyarrow scikit-learn

## 3. Generate and verify the shared processed dataset

In [3]:
!python -m src.prepare_data

from pathlib import Path
DATA_PATH = Path("data/processed/loans_clean.parquet")
assert DATA_PATH.is_file(), f"Dataset was not created: {DATA_PATH}"
print("Verified dataset:", DATA_PATH.resolve())
print("Size (MB):", round(DATA_PATH.stat().st_size / (1024 * 1024), 2))

reading: https://raw.githubusercontent.com/tiwvis/MLBD-Projects/main/data_sets/LoansData_sample.csv.gz
saved: /content/loan-default-mlbd/data/processed/loans_clean.parquet (86141, 71)
        size      mean
split                 
test   12922  0.187587
train  60298  0.187585
val    12921  0.187602
Verified dataset: /content/loan-default-mlbd/data/processed/loans_clean.parquet
Size (MB): 6.87


## 4. Create the Python 3.10 Flash environment
This step is required in every fresh Colab runtime. It must finish before the training cell is run.

In [4]:
!uv python install 3.10
!uv venv --python 3.10 .venv_flash
!uv pip install --python .venv_flash/bin/python "lightning-flash[tabular]==0.8.2" "setuptools<81" requests pyarrow

Installed Python 3.10.22 in 1.29s
 + cpython-3.10.22-linux-x86_64-gnu (python3.10)
Using CPython 3.10.22
Creating virtual environment at: .venv_flash
Activate with: source .venv_flash/bin/activate
Using Python 3.10.22 environment at: .venv_flash
Resolved 113 packages in 3.58s
Prepared 113 packages in 56.06s
Installed 113 packages in 597ms
 + absl-py==2.5.1
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + alembic==1.20.1
 + asttokens==3.0.2
 + async-timeout==5.0.1
 + attrs==26.1.0
 + autopage==0.6.0
 + backports-strenum==1.3.1
 + category-encoders==2.6.4
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cliff==4.14.0
 + cloudpickle==3.1.2
 + cmaes==0.13.1
 + cmd2==3.5.1
 + colorlog==6.12.0
 + comm==0.2.3
 + contourpy==1.3.2
 + cycler==0.12.1
 + decorator==5.3.1
 + docstring-parser==0.18.0
 + einops==0.6.1
 + exceptiongroup==1.3.1
 + executing==2.3.0
 + fonttools==4.65.0
 + formulaic==1.2.2
 + frozenlist==1.8.0
 + fsspec==2026.9.0
 + greenlet==3.5

## 5. Verify the exact Flash environment

In [5]:
! .venv_flash/bin/python -c "import sys, torch, flash, pytorch_lightning; from flash.tabular import TabularClassificationData, TabularClassifier; print('Python:', sys.version); print('PyTorch:', torch.__version__); print('Flash:', flash.__version__); print('PyTorch Lightning:', pytorch_lightning.__version__); print('Flash imports successful')"


/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/pytorch_tabular/models/mixture_density/mdn.py:25: UserWarning: Wandb not installed. WandbLogger will 

## 6. Create the training script
The script uses the team's `load_splits` and `make_nn_data` definitions. It fits only on the training split and uses validation during training. It does not evaluate the test split.

In [6]:
%%writefile flash_train.py
import os
import json
import time
import torch
import pytorch_lightning as pl

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from flash.tabular import TabularClassificationData, TabularClassifier
from src.data import load_splits, make_nn_data

SEED = 42
pl.seed_everything(SEED)

DATA_PATH = "data/processed/loans_clean.parquet"
tr, va, te = load_splits(DATA_PATH)
_, num_cols, cat_cols, shared_cards = make_nn_data(tr, va, te)

def prepare_frame(df):
    frame = df[num_cols + cat_cols + ["default"]].copy()
    for col in cat_cols:
        frame[col] = frame[col].astype(str)
    frame["default"] = frame["default"].astype(str)
    return frame

train_df = prepare_frame(tr)
val_df = prepare_frame(va)
test_df = prepare_frame(te)

dm = TabularClassificationData.from_data_frame(
    categorical_fields=cat_cols,
    numerical_fields=num_cols,
    target_fields="default",
    train_data_frame=train_df,
    val_data_frame=val_df,
    test_data_frame=test_df,
    batch_size=1024,
)

# Flash builds category codes from the training DataFrame.
flash_codes = dm.parameters["codes"]
cat_dims = [len(flash_codes[col]) + 1 for col in cat_cols]
embedding_sizes = [
    (dim, min(50, max(1, (dim + 1) // 2)))
    for dim in cat_dims
]

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Shared category cardinalities:", shared_cards)
print("Flash category code counts:", [len(flash_codes[c]) for c in cat_cols])
print("TabNet embedding dimensions:", cat_dims)
print("Classes:", dm.labels, "Number of classes:", dm.num_classes)

model = TabularClassifier(
    parameters=dm.parameters,
    embedding_sizes=embedding_sizes,
    cat_dims=cat_dims,
    num_features=len(num_cols),
    num_classes=dm.num_classes,
    backbone="tabnet",
    learning_rate=0.02,
)

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)

checkpoint = ModelCheckpoint(
    dirpath="checkpoints",
    filename="flash-tabnet-{epoch:02d}",
    save_top_k=1,
    monitor=None,
)

trainer = Trainer(
    max_epochs=30,
    accelerator="auto",
    devices=1,
    callbacks=[checkpoint],
    logger=False,
    enable_progress_bar=True,
    log_every_n_steps=20,
)

start = time.time()
trainer.fit(model, datamodule=dm)
train_seconds = time.time() - start

metadata = {
    "framework": "Lightning Flash",
    "backbone": "TabNet",
    "seed": SEED,
    "max_epochs": 30,
    "batch_size": 1024,
    "learning_rate": 0.02,
    "train_seconds": train_seconds,
    "checkpoint": checkpoint.best_model_path,
    "validation_metrics_logged": {
        str(k): float(v.detach().cpu()) if hasattr(v, "detach") else str(v)
        for k, v in trainer.callback_metrics.items()
    },
}

with open("reports/flash_training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Training completed.")
print("Training seconds:", round(train_seconds, 2))
print("Validation metrics logged:", metadata["validation_metrics_logged"])
print("Checkpoint:", checkpoint.best_model_path)
print("Metadata: reports/flash_training_metadata.json")


Writing flash_train.py


## 7. Run Lightning Flash training
This invokes the script with the Python 3.10 environment created in Step 4. Do not run `flash_train.py` with the normal Colab Python kernel.

In [7]:
! .venv_flash/bin/python flash_train.py

/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/pytorch_tabular/models/mixture_density/mdn.py:25: UserWarning: Wandb not installed. WandbLogger will 

## 8. Inspect saved training metadata
Run this only if training completed successfully.

In [11]:
import json
from pathlib import Path

metadata_path = Path("reports/flash_training_metadata.json")
if metadata_path.exists():
    print(json.dumps(json.loads(metadata_path.read_text()), indent=2))
else:
    print("No metadata file: training did not complete successfully.")

{
  "framework": "Lightning Flash",
  "backbone": "TabNet",
  "seed": 42,
  "max_epochs": 30,
  "batch_size": 1024,
  "learning_rate": 0.02,
  "train_seconds": 220.9979829788208,
  "checkpoint": "checkpoints/flash-tabnet-epoch=29.ckpt",
  "validation_metrics_logged": {
    "train_loss": 0.4548463821411133,
    "valid_loss": 0.4397141635417938,
    "valid_accuracy": 0.8113149404525757,
    "train_accuracy": 0.8148403763771057
  }
}


## 9. Export the artifacts
Download the notebook, training metadata, and checkpoint only if the corresponding files exist. Do not create a `results_flash.json` until predictions have been evaluated with the team's shared `src.metrics.evaluate` function.

In [13]:
from google.colab import files
from pathlib import Path
import json

files.download("reports/flash_training_metadata.json")

checkpoint_dir = Path("checkpoints")
checkpoints = list(checkpoint_dir.glob("*.ckpt"))

if checkpoints:
    for checkpoint in checkpoints:
        files.download(str(checkpoint))
else:
    print("No checkpoint found.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os

print("Current directory:", os.getcwd())
print("Flash environment:", os.path.exists(".venv_flash/bin/python"))
print("Dataset:", os.path.exists("data/processed/loans_clean.parquet"))
print("Checkpoint:", os.path.exists("checkpoints/flash-tabnet-epoch=29.ckpt"))
print("Metadata:", os.path.exists("reports/flash_training_metadata.json"))

Current directory: /content/loan-default-mlbd
Flash environment: True
Dataset: True
Checkpoint: True
Metadata: True


In [20]:
! .venv_flash/bin/python -c "from flash.tabular import TabularClassifier; m=TabularClassifier.load_from_checkpoint('checkpoints/flash-tabnet-epoch=29.ckpt'); print('Checkpoint loaded'); print('Outputs:', m.outputs); print('Model class:', type(m).__name__)"

/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/pytorch_tabular/models/mixture_density/mdn.py:25: UserWarning: Wandb not installed. WandbLogger will 

In [31]:
%%writefile evaluate_flash_corrected.py

import json
import numpy as np

from flash import Trainer
from flash.tabular import TabularClassificationData, TabularClassifier
from src.data import load_splits, make_nn_data
from src.metrics import evaluate

# Load the shared dataset splits.
tr, va, te = load_splits("data/processed/loans_clean.parquet")
_, num_cols, cat_cols, _ = make_nn_data(tr, va, te)


def prepare(df):
    result = df[num_cols + cat_cols].copy()
    for col in cat_cols:
        result[col] = result[col].astype(str)
    return result


# Recreate training preprocessing parameters.
train_df = prepare(tr)
train_df["default"] = tr["default"].astype(str)

dm_train = TabularClassificationData.from_data_frame(
    categorical_fields=cat_cols,
    numerical_fields=num_cols,
    target_fields="default",
    train_data_frame=train_df,
    batch_size=1024
)

# Load the trained checkpoint.
model = TabularClassifier.load_from_checkpoint(
    "checkpoints/flash-tabnet-epoch=29.ckpt"
)

# Use the Flash Trainer, not pytorch_lightning.Trainer.
trainer = Trainer(
    accelerator="cpu",
    devices=1,
    logger=False,
    enable_progress_bar=False
)


def get_probabilities(df):
    dm = TabularClassificationData.from_data_frame(
        predict_data_frame=prepare(df),
        parameters=dm_train.parameters,
        batch_size=1024
    )

    batches = trainer.predict(
        model,
        datamodule=dm,
        output="probabilities"
    )

    return np.concatenate(
        [np.asarray(batch) for batch in batches],
        axis=0
    )


# Predict validation and test splits.
val_probs = get_probabilities(va)
test_probs = get_probabilities(te)

# Verify probability outputs.
assert val_probs.shape == (len(va), 2), val_probs.shape
assert test_probs.shape == (len(te), 2), test_probs.shape
assert np.all(np.isfinite(val_probs))
assert np.all(np.isfinite(test_probs))
assert np.all((val_probs >= 0) & (val_probs <= 1))
assert np.all((test_probs >= 0) & (test_probs <= 1))
assert np.allclose(val_probs.sum(axis=1), 1, atol=1e-4)
assert np.allclose(test_probs.sum(axis=1), 1, atol=1e-4)

print("Validation probabilities:", val_probs.shape)
print("Test probabilities:", test_probs.shape)
print("Validation probability range:", val_probs.min(), val_probs.max())

# Evaluate default class (1).
val_results = evaluate(va["default"], val_probs[:, 1])
test_results = evaluate(te["default"], test_probs[:, 1])

print("\nValidation metrics:", val_results)
print("\nTest metrics:", test_results)

# Save the final results.
results = {
    "framework": "Lightning Flash",
    "params": {
        "backbone": "TabNet",
        "max_epochs": 30,
        "batch_size": 1024,
        "learning_rate": 0.02,
        "seed": 42
    },
    "train_seconds": 220.9979829788208,
    "val": val_results,
    "test": test_results
}

with open("reports/results_flash.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nSaved reports/results_flash.json")


Overwriting evaluate_flash_corrected.py


In [32]:
! .venv_flash/bin/python evaluate_flash_corrected.py

/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/flash/__init__.py:21: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/content/loan-default-mlbd/.venv_flash/lib/python3.10/site-packages/pytorch_tabular/models/mixture_density/mdn.py:25: UserWarning: Wandb not installed. WandbLogger will 

In [33]:
from google.colab import files
files.download("reports/results_flash.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>